**FAMA FRENCH 3 FACTOR MODEL**

In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

In [24]:
import pandas_datareader.data as web

try:
    # Fetching the daily research factors
    factors = web.DataReader('F-F_Research_Data_Factors_daily', 'famafrench', start='2025-01-01', end='2026-01-01')[0]
    factors['Mkt-RF'] = factors['Mkt-RF']/100
    factors['SMB'] = factors['SMB']/100
    factors['HML'] = factors['HML']/100
    factors['RF'] = factors ['RF']/100
except Exception as e:
    print(f"Extraction failed due to: {e}")

factors


/tmp/ipykernel_1979/1037696467.py:5: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  factors = web.DataReader('F-F_Research_Data_Factors_daily', 'famafrench', start='2025-01-01', end='2026-01-01')[0]


,Mkt-RF,SMB,HML,RF
Date,,,,
2025-01-02,-0.0016,0.0010,-0.0031,0.0002
2025-01-03,0.0134,0.0061,-0.0093,0.0002
2025-01-06,0.0055,-0.0037,-0.0034,0.0002
2025-01-07,-0.0118,-0.0037,0.0084,0.0002
2025-01-08,0.0010,-0.0059,-0.0009,0.0002
...,...,...,...,...
2025-12-24,0.0029,0.0004,0.0001,0.0002
2025-12-26,-0.0006,-0.0032,0.0009,0.0002
2025-12-29,-0.0041,-0.0016,0.0006,0.0002


In [15]:
ticker = 'MA' #MASTERCARD
start_date = '2025-01-01'
end_date = '2026-01-01'

df = yf.download(ticker,start_date,end_date)
df

/tmp/ipykernel_1979/3779572656.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker,start_date,end_date)
[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,MA,MA,MA,MA,MA
Date,,,,,
2025-01-02,517.746887,525.834178,513.049057,524.476385,2818800
2025-01-03,516.716064,519.788488,513.068851,517.875686,1627500
2025-01-06,507.370117,516.716115,506.458329,516.329574,2920800
2025-01-07,506.844879,511.730990,506.527722,509.104570,2211800
2025-01-08,511.800415,512.841051,506.973767,507.479204,2456700
...,...,...,...,...,...
2025-12-24,577.588379,580.419221,575.146238,575.395435,1058900
2025-12-26,577.737793,579.332689,576.820766,577.887335,972300


In [16]:
stock_returns = df['Close']['MA'].pct_change()
stock_returns

,MA
Date,
2025-01-02,NaN
2025-01-03,-0.001991
2025-01-06,-0.018087
2025-01-07,-0.001035
2025-01-08,0.009777
...,...
2025-12-24,0.005379
2025-12-26,0.000259
2025-12-29,-0.002933


In [17]:
risk_free_rate = 0.0002

In [18]:
daily_excess_return = stock_returns - factors['RF']
daily_excess_return

,0
Date,
2025-01-02,NaN
2025-01-03,-0.002191
2025-01-06,-0.018287
2025-01-07,-0.001235
2025-01-08,0.009577
...,...
2025-12-24,0.005179
2025-12-26,0.000059
2025-12-29,-0.003133


In [35]:
from statsmodels import *

In [23]:
daily_excess_return.dropna(inplace=True)
factors.dropna(inplace=True)
daily_excess_return.isnull().sum().sum()
factors.isnull().sum().sum()

np.int64(0)

In [38]:
df = pd.DataFrame({
    'Daily Excess Return': daily_excess_return,
    'Mkt-RF': factors['Mkt-RF'],
    'SMB': factors['SMB'],
    'HML': factors['HML']
})
df.dropna(inplace=True)

In [39]:
y = df['Daily Excess Return']
x = df.drop('Daily Excess Return',axis = 1)
y.dropna(inplace=True)
x

,Mkt-RF,SMB,HML
Date,,,
2025-01-03,0.0134,0.0061,-0.0093
2025-01-06,0.0055,-0.0037,-0.0034
2025-01-07,-0.0118,-0.0037,0.0084
2025-01-08,0.0010,-0.0059,-0.0009
2025-01-10,-0.0155,-0.0090,-0.0018
...,...,...,...
2025-12-24,0.0029,0.0004,0.0001
2025-12-26,-0.0006,-0.0032,0.0009
2025-12-29,-0.0041,-0.0016,0.0006


In [40]:
import statsmodels.api as sm
X_const = sm.add_constant(x)
model = sm.OLS(y, X_const).fit()
print(model.summary())

                             OLS Regression Results                            
Dep. Variable:     Daily Excess Return   R-squared:                       0.480
Model:                             OLS   Adj. R-squared:                  0.474
Method:                  Least Squares   F-statistic:                     75.46
Date:                 Fri, 12 Jun 2026   Prob (F-statistic):           1.34e-34
Time:                         05:58:26   Log-Likelihood:                 785.34
No. Observations:                  249   AIC:                            -1563.
Df Residuals:                      245   BIC:                            -1549.
Df Model:                            3                                         
Covariance Type:             nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0005      0.001     -0.68

In [ ]:
#Beta 1 is below 1 (0.9163) which means  Mastercard moves ~0.92% for every 1% market move — slightly less volatile than the market, indicating low but near-average systematic risk.
#A negative β₂ means Mastercard behaves like a large-cap stock — it moves against the small-cap factor. Which makes sense — Mastercard is one of the largest companies in the world.
#Beta 3 (HML - High Minus Low) is 0.5555. HML is value stocks minus growth stocks. A positive β₃ means Mastercard loads on the value factor — it behaves more like a value stock than a growth stock. Surprising for a payments company, but the data says so.
# alpha of -0.0005 is not meaningfully different from zero. Mastercard generates no excess return beyond what the three factors explain.
#R2 of 0.48 means the three factors explain 48% of the daily variation in Mastercard's excess returns. The other 52% is driven by company-specific factors not captured by this model.